# Contribution Tasks: sklearn Transformer + Batch Processing + Multi-Class

This notebook covers the three code-contribution tasks for the `llm-feature-gen` coursework, applied to the **Netflix Titles** dataset.

| Task | Points |
|---|---:|
| Integrate scikit-learn interface + tests | 1 |
| Add batch processing + caching + tests | 1 |
| Generalize class handling (N classes) + tests | 1 |

All code changes below belong in a fork of [https://github.com/JuliaYershova/llm-feature-gen](https://github.com/JuliaYershova/llm-feature-gen).

In [1]:
%pip install -U -qq llm-feature-gen scikit-learn pandas

import json, re, shutil, hashlib, random, tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from llm_feature_gen.discover import discover_features_from_texts
from llm_feature_gen.generate import generate_features_from_texts
from llm_feature_gen.providers.local_provider import LocalProvider

random_seed = 42
DROP = ['File', 'Class', 'raw_llm_output', 'split']

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Patch for qwen3 thinking models ─────────────────────────────
# qwen3 outputs <think>...</think> before JSON which breaks the library.
# This cell must run before any discover/generate call.
import re, json as _json, time
import openai
from openai import BadRequestError
import llm_feature_gen.providers.local_provider as _lp

def _fixed_chat_json(self, deployment_name, system_prompt, user_content, json_mode=False):
    if json_mode and 'JSON' not in system_prompt:
        system_prompt += ' Respond in strict JSON format.'
    kwargs = {}
    if json_mode:
        kwargs['response_format'] = {'type': 'json_object'}
    backoff = 2
    for attempt in range(self.max_retries):
        try:
            resp = self.client.chat.completions.create(
                model=deployment_name,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_content},
                ],
                temperature=self.temperature,
                max_tokens=self.max_tokens,
                **kwargs,
            )
            text = resp.choices[0].message.content or ''
            # Strip thinking blocks
            text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
            try:
                return _json.loads(text)
            except Exception:
                extracted = self._extract_json(text)
                if extracted:
                    if isinstance(extracted, list):
                        return {'features': extracted}
                    return extracted
                return {'features': text}
        except BadRequestError as e:
            if json_mode and 'json_object' in str(e):
                json_mode = False
                kwargs.pop('response_format', None)
                continue
            raise e
        except openai.RateLimitError as e:
            if attempt < self.max_retries - 1:
                time.sleep(backoff)
                backoff *= 2
                continue
            raise e
        except Exception as e:
            raise e
    raise RuntimeError('Unknown failure: unable to get response.')

_lp.LocalProvider._chat_json = _fixed_chat_json
print('qwen3 patch applied.')

from llm_feature_gen.providers.local_provider import LocalProvider

provider = LocalProvider(
    base_url='https://litellm.vse.cz/',
    api_key='sk-ekE7TcLJMcj_kOj2TACvyA',
    default_text_model='qwen3.6-35b',
    temperature=0.2,
)

print('Provider ready.')


qwen3 patch applied.
Provider ready.


---
## Task 1 — scikit-learn Pipeline Integration

**What:** `LLMFeatureTransformer` — a `TransformerMixin` that wraps discover + generate so it works as a drop-in sklearn transformer in any `Pipeline`.

**In the fork:** `llm_feature_gen/sklearn_transformer.py` + `tests/test_sklearn_transformer.py`

In [3]:
# llm_feature_gen/sklearn_transformer.py

class LLMFeatureTransformer(BaseEstimator, TransformerMixin):
    """
    sklearn Transformer wrapping llm-feature-gen discover + generate.

    Parameters
    ----------
    provider           : LLM provider (LocalProvider or compatible)
    classes            : list of class label strings
    discovery_fraction : fraction of training texts used for discovery
    random_state       : random seed
    """

    def __init__(self, provider, classes: List[str],
                 discovery_fraction: float = 0.1,
                 random_state: int = 42):
        self.provider           = provider
        self.classes            = classes
        self.discovery_fraction = discovery_fraction
        self.random_state       = random_state

    def fit(self, X: List[str], y: List[str]):
        self._workdir_   = Path(tempfile.mkdtemp())
        disc_dir         = self._workdir_ / 'discover'
        disc_dir.mkdir()
        self._output_dir = self._workdir_ / 'outputs'
        self._output_dir.mkdir()

        random.seed(self.random_state)
        n   = max(4, int(len(X) * self.discovery_fraction))
        idx = random.sample(range(len(X)), min(n, len(X)))
        for i, j in enumerate(idx, 1):
            (disc_dir / f'sample_{i}.txt').write_text(X[j], encoding='utf-8')

        self.discovered_features_ = discover_features_from_texts(
            texts_or_file=disc_dir,
            provider=self.provider,
            as_set=True,
            output_dir=self._output_dir,
            output_filename='discovered_text_features.json',
        )
        return self

    def transform(self, X: List[str], y: Optional[List[str]] = None):
        gen_in  = self._workdir_ / 'gen_in'
        gen_out = self._workdir_ / 'gen_out'
        for p in [gen_in, gen_out]:
            if p.exists(): shutil.rmtree(p)

        if y is not None:
            for label, text in zip(y, X):
                d = gen_in / str(label)
                d.mkdir(parents=True, exist_ok=True)
                n = len(list(d.glob('*.txt')))
                (d / f'item_{n+1}.txt').write_text(text, encoding='utf-8')
            classes = self.classes
        else:
            d = gen_in / '_unknown'
            d.mkdir(parents=True, exist_ok=True)
            for i, text in enumerate(X, 1):
                (d / f'item_{i}.txt').write_text(text, encoding='utf-8')
            classes = ['_unknown']

        csv_paths = generate_features_from_texts(
            root_folder=gen_in,
            discovered_features_path=self._output_dir / 'discovered_text_features.json',
            provider=self.provider,
            classes=classes,
            output_dir=gen_out,
            merge_to_single_csv=True,
            merged_csv_name='merged.csv',
        )
        merged = pd.read_csv(csv_paths['__merged__'])
        merged = merged.drop(columns=[c for c in DROP if c in merged.columns])
        return pd.get_dummies(merged, dtype=int).values


print('LLMFeatureTransformer defined.')
print('''
Usage:
    pipe = Pipeline([
        ("llm", LLMFeatureTransformer(provider=provider, classes=["Movie", "TV Show"])),
        ("clf", DecisionTreeClassifier(max_depth=3)),
    ])
    pipe.fit(train_texts, train_labels)
    preds = pipe.predict(test_texts)
''')

LLMFeatureTransformer defined.

Usage:
    pipe = Pipeline([
        ("llm", LLMFeatureTransformer(provider=provider, classes=["Movie", "TV Show"])),
        ("clf", DecisionTreeClassifier(max_depth=3)),
    ])
    pipe.fit(train_texts, train_labels)
    preds = pipe.predict(test_texts)



In [4]:
# tests/test_sklearn_transformer.py
# Copy this file into your fork under tests/

SKLEARN_TESTS = '''
"""Tests for LLMFeatureTransformer."""
import json, tempfile
from pathlib import Path
from unittest.mock import MagicMock, patch
import numpy as np
import pandas as pd
import pytest
from llm_feature_gen.sklearn_transformer import LLMFeatureTransformer

FEATURES = ["implies_ongoing_narrative", "self_contained_plot", "mentions_seasons"]
ROWS = [
    {"File": "a.txt", "Class": "Movie",   "implies_ongoing_narrative": "no",  "self_contained_plot": "yes", "mentions_seasons": "no"},
    {"File": "b.txt", "Class": "TV Show",  "implies_ongoing_narrative": "yes", "self_contained_plot": "no",  "mentions_seasons": "yes"},
]

def _csv(tmp_path):
    p = tmp_path / "merged.csv"
    pd.DataFrame(ROWS).to_csv(p, index=False)
    return str(p)

def test_fit_calls_discovery(tmp_path):
    with patch("llm_feature_gen.sklearn_transformer.discover_features_from_texts", return_value=FEATURES) as md, \\
         patch("llm_feature_gen.sklearn_transformer.generate_features_from_texts", return_value={"__merged__": _csv(tmp_path)}):
        t = LLMFeatureTransformer(provider=MagicMock(), classes=["Movie", "TV Show"])
        t.fit(["A movie.", "A series."], ["Movie", "TV Show"])
        assert md.called and t.discovered_features_ == FEATURES

def test_transform_returns_ndarray(tmp_path):
    with patch("llm_feature_gen.sklearn_transformer.discover_features_from_texts", return_value=FEATURES), \\
         patch("llm_feature_gen.sklearn_transformer.generate_features_from_texts", return_value={"__merged__": _csv(tmp_path)}):
        t = LLMFeatureTransformer(provider=MagicMock(), classes=["Movie", "TV Show"])
        t.fit(["A movie.", "A series."], ["Movie", "TV Show"])
        result = t.transform(["A movie.", "A series."], ["Movie", "TV Show"])
        assert isinstance(result, np.ndarray) and result.shape[0] == 2

def test_pipeline_compatible(tmp_path):
    from sklearn.pipeline import Pipeline
    from sklearn.tree import DecisionTreeClassifier
    with patch("llm_feature_gen.sklearn_transformer.discover_features_from_texts", return_value=FEATURES), \\
         patch("llm_feature_gen.sklearn_transformer.generate_features_from_texts", return_value={"__merged__": _csv(tmp_path)}):
        pipe = Pipeline([("llm", LLMFeatureTransformer(provider=MagicMock(), classes=["Movie", "TV Show"])), ("clf", DecisionTreeClassifier(max_depth=2))])
        pipe.fit(["A movie.", "A series."], ["Movie", "TV Show"])
        assert len(pipe.predict(["A movie.", "A series."])) == 2

def test_get_params():
    t = LLMFeatureTransformer(provider=MagicMock(), classes=["Movie", "TV Show"], discovery_fraction=0.2)
    assert t.get_params()["discovery_fraction"] == 0.2
'''

print(SKLEARN_TESTS)


"""Tests for LLMFeatureTransformer."""
import json, tempfile
from pathlib import Path
from unittest.mock import MagicMock, patch
import numpy as np
import pandas as pd
import pytest
from llm_feature_gen.sklearn_transformer import LLMFeatureTransformer

FEATURES = ["implies_ongoing_narrative", "self_contained_plot", "mentions_seasons"]
ROWS = [
    {"File": "a.txt", "Class": "Movie",   "implies_ongoing_narrative": "no",  "self_contained_plot": "yes", "mentions_seasons": "no"},
    {"File": "b.txt", "Class": "TV Show",  "implies_ongoing_narrative": "yes", "self_contained_plot": "no",  "mentions_seasons": "yes"},
]

def _csv(tmp_path):
    p = tmp_path / "merged.csv"
    pd.DataFrame(ROWS).to_csv(p, index=False)
    return str(p)

def test_fit_calls_discovery(tmp_path):
    with patch("llm_feature_gen.sklearn_transformer.discover_features_from_texts", return_value=FEATURES) as md, \
         patch("llm_feature_gen.sklearn_transformer.generate_features_from_texts", return_value={"__merged

---
## Task 2 — Batch Processing + Caching

**What:** The original generate step calls the LLM once per file sequentially. For 8,800+ Netflix titles this is extremely slow. This adds configurable batching and SHA-256 disk caching.

**In the fork:** `llm_feature_gen/batch_generate.py` + `tests/test_batch_processing.py`

In [5]:
# llm_feature_gen/batch_generate.py

def _cache_key(text: str, features: List[str]) -> str:
    payload = json.dumps({'text': text, 'features': sorted(features)}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()


def _extract_json(raw: str):
    """Safely extract JSON from LLM output that may include <think> blocks or markdown."""
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    raw = re.sub(r'^```(?:json)?\s*', '', raw).rstrip('`').strip()
    m = re.search(r'(\[.*\]|\{.*\})', raw, re.DOTALL)
    return json.loads(m.group(1) if m else raw)


def batch_generate_features(
    texts: List[str],
    features: List[str],
    provider,
    batch_size: int = 10,
    cache_dir: Optional[Path] = None,
) -> List[Dict[str, Any]]:
    """
    Generate feature values for a list of texts in configurable batches,
    with optional persistent disk caching.

    Parameters
    ----------
    texts      : list of raw text strings
    features   : feature names from the discovery step
    provider   : LLM provider instance
    batch_size : number of texts per LLM call
    cache_dir  : optional directory for persistent result caching

    Returns
    -------
    One dict per text mapping feature name -> value.
    """
    if cache_dir is not None:
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)

    results    = [None] * len(texts)
    to_process = []

    for i, text in enumerate(texts):
        if cache_dir is not None:
            path = cache_dir / f'{_cache_key(text, features)}.json'
            if path.exists():
                results[i] = json.loads(path.read_text())
                continue
        to_process.append((i, text))

    for start in range(0, len(to_process), batch_size):
        batch       = to_process[start: start + batch_size]
        batch_texts = [t for _, t in batch]

        prompt = (
            f'You will receive {len(batch_texts)} text snippets numbered 1 to {len(batch_texts)}.\n'
            f'For each snippet assign a value to every feature: {json.dumps(features)}\n\n'
            'Return ONLY a JSON array, one object per snippet, in order. '
            'Each object must have exactly the feature names as keys. No markdown, no explanation.\n\n'
            + '\n'.join(f'{j+1}. {t}' for j, t in enumerate(batch_texts))
        )

        raw = provider.complete(prompt)
        try:
            batch_results = _extract_json(raw)
            if not isinstance(batch_results, list):
                raise ValueError
        except Exception:
            batch_results = [{f: None for f in features} for _ in batch_texts]

        for (orig_idx, text), fdict in zip(batch, batch_results):
            results[orig_idx] = fdict
            if cache_dir is not None:
                (cache_dir / f'{_cache_key(text, features)}.json').write_text(json.dumps(fdict))

    return results


df_netflix = pd.read_csv('netflix_titles.csv')
n = len(df_netflix)
print(f'batch_generate_features ready.')
print(f'Netflix dataset: {n} titles')
print(f'  Sequential : {n} LLM calls')
print(f'  Batched=10 : {n // 10 + 1} LLM calls  ({round(n / (n//10+1))}x fewer)')

batch_generate_features ready.
Netflix dataset: 8807 titles
  Sequential : 8807 LLM calls
  Batched=10 : 881 LLM calls  (10x fewer)


In [6]:
# tests/test_batch_processing.py

BATCH_TESTS = '''
"""Tests for batch_generate_features."""
import json, tempfile
from pathlib import Path
from unittest.mock import MagicMock
import pytest
from llm_feature_gen.batch_generate import batch_generate_features, _cache_key

FEATURES = ["implies_ongoing_narrative", "self_contained_plot"]
TEXTS    = ["A war epic.", "A love story.", "A thriller series."]

def make_provider(responses):
    p = MagicMock()
    p.complete.side_effect = [json.dumps(r) for r in responses]
    return p

def test_returns_one_dict_per_text():
    resp = [[{"implies_ongoing_narrative": "no",  "self_contained_plot": "yes"},
             {"implies_ongoing_narrative": "no",  "self_contained_plot": "yes"},
             {"implies_ongoing_narrative": "yes", "self_contained_plot": "no"}]]
    results = batch_generate_features(TEXTS, FEATURES, make_provider(resp), batch_size=10)
    assert len(results) == 3 and all(isinstance(r, dict) for r in results)

def test_batch_size_controls_llm_calls():
    resp = [[{"implies_ongoing_narrative": "no", "self_contained_plot": "yes"}, {"implies_ongoing_narrative": "no", "self_contained_plot": "yes"}],
            [{"implies_ongoing_narrative": "yes", "self_contained_plot": "no"}]]
    p = make_provider(resp)
    batch_generate_features(TEXTS, FEATURES, p, batch_size=2)
    assert p.complete.call_count == 2

def test_cache_prevents_recomputation(tmp_path):
    resp = [[{"implies_ongoing_narrative": "no", "self_contained_plot": "yes"},
             {"implies_ongoing_narrative": "no", "self_contained_plot": "yes"},
             {"implies_ongoing_narrative": "yes", "self_contained_plot": "no"}]]
    batch_generate_features(TEXTS, FEATURES, make_provider(resp), cache_dir=tmp_path)
    p2 = MagicMock()
    batch_generate_features(TEXTS, FEATURES, p2, cache_dir=tmp_path)
    assert p2.complete.call_count == 0

def test_malformed_response_returns_none_values():
    p = MagicMock()
    p.complete.return_value = "not valid json"
    assert batch_generate_features(["text"], FEATURES, p)[0] == {f: None for f in FEATURES}

def test_cache_key_order_independent():
    assert _cache_key("hello", ["a", "b"]) == _cache_key("hello", ["b", "a"])
'''

print(BATCH_TESTS)


"""Tests for batch_generate_features."""
import json, tempfile
from pathlib import Path
from unittest.mock import MagicMock
import pytest
from llm_feature_gen.batch_generate import batch_generate_features, _cache_key

FEATURES = ["implies_ongoing_narrative", "self_contained_plot"]
TEXTS    = ["A war epic.", "A love story.", "A thriller series."]

def make_provider(responses):
    p = MagicMock()
    p.complete.side_effect = [json.dumps(r) for r in responses]
    return p

def test_returns_one_dict_per_text():
    resp = [[{"implies_ongoing_narrative": "no",  "self_contained_plot": "yes"},
             {"implies_ongoing_narrative": "no",  "self_contained_plot": "yes"},
             {"implies_ongoing_narrative": "yes", "self_contained_plot": "no"}]]
    results = batch_generate_features(TEXTS, FEATURES, make_provider(resp), batch_size=10)
    assert len(results) == 3 and all(isinstance(r, dict) for r in results)

def test_batch_size_controls_llm_calls():
    resp = [[{"implies_ongoing_n

---
## Task 3 — Generalise Class Handling (N Classes)

**What:** The original library assumes exactly two classes. This generalises it to N classes.

**Demonstrated with:** 3-class Netflix task — **Kids / Teen / Adult** audience classification from rating.

**In the fork:** discovery prompt receives all class names dynamically; generate iterates all class subfolders. Tests in `tests/test_multiclass.py`.

In [7]:
rating_map = {
    'TV-Y': 'Kids', 'TV-Y7': 'Kids', 'TV-Y7-FV': 'Kids',
    'TV-G': 'Kids', 'PG': 'Kids', 'G': 'Kids',
    'TV-PG': 'Teen', 'PG-13': 'Teen', 'TV-14': 'Teen',
    'TV-MA': 'Adult', 'R': 'Adult', 'NR': 'Adult',
}

df_full = pd.read_csv('netflix_titles.csv')
df_multi = (
    df_full[['description', 'rating']].dropna()
    .assign(class_name=lambda d: d['rating'].map(rating_map))
    .dropna(subset=['class_name'])
    .rename(columns={'description': 'text'})
    [['text', 'class_name']].reset_index(drop=True)
)

multi_classes = ['Kids', 'Teen', 'Adult']
print('3-class distribution:')
print(df_multi['class_name'].value_counts())

3-class distribution:
class_name
Adult    4086
Teen     3513
Kids     1195
Name: count, dtype: int64


In [8]:
# ── Small sample to keep LLM calls fast ─────────────────────────
N_TRAIN_PER_CLASS = 40
N_TEST_PER_CLASS  = 20

# Sample N per class — works with all pandas versions
train_parts = []
for cls in multi_classes:
    subset = df_multi[df_multi['class_name'] == cls]
    train_parts.append(subset.sample(n=min(N_TRAIN_PER_CLASS, len(subset)), random_state=random_seed))
multi_train = pd.concat(train_parts).reset_index(drop=True)

remaining = df_multi.drop(multi_train.index)
test_parts = []
for cls in multi_classes:
    subset = remaining[remaining['class_name'] == cls]
    test_parts.append(subset.sample(n=min(N_TEST_PER_CLASS, len(subset)), random_state=random_seed))
multi_test = pd.concat(test_parts).reset_index(drop=True)

print(pd.concat([
    multi_train.assign(split='train'),
    multi_test.assign(split='test'),
]).groupby(['split', 'class_name']).size())
print(f'Total LLM calls needed: {len(multi_train) + len(multi_test)} files')


split  class_name
test   Adult         20
       Kids          20
       Teen          20
train  Adult         40
       Kids          40
       Teen          40
dtype: int64
Total LLM calls needed: 180 files


In [9]:
multi_dir        = Path('netflix_multiclass_data')
multi_disc_dir   = multi_dir / 'discover_texts'
multi_train_dir  = multi_dir / 'train_texts'
multi_test_dir   = multi_dir / 'test_texts'
multi_output_dir = multi_dir / 'outputs'

N_DISCOVER_PER_CLASS = 5

for p in [multi_disc_dir, multi_train_dir, multi_test_dir, multi_output_dir]:
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

for cls in multi_classes:
    (multi_train_dir / cls).mkdir(parents=True, exist_ok=True)
    (multi_test_dir  / cls).mkdir(parents=True, exist_ok=True)

for cls in multi_classes:
    tr   = multi_train[multi_train['class_name'] == cls].reset_index(drop=True)
    te   = multi_test [multi_test ['class_name'] == cls].reset_index(drop=True)
    safe = cls.replace(' ', '_')
    for i, text in enumerate(
        tr.sample(n=min(N_DISCOVER_PER_CLASS, len(tr)), random_state=random_seed)['text'], 1
    ):
        (multi_disc_dir / f'{safe}_discover_{i}.txt').write_text(text, encoding='utf-8')
    for i, text in enumerate(tr['text'], 1):
        (multi_train_dir / cls / f'{safe}_train_{i}.txt').write_text(text, encoding='utf-8')
    for i, text in enumerate(te['text'], 1):
        (multi_test_dir / cls / f'{safe}_test_{i}.txt').write_text(text, encoding='utf-8')

print('Multi-class folder structure ready.')
print('Discovery files:', len(list(multi_disc_dir.glob('*.txt'))))
for cls in multi_classes:
    print(cls, 'train:', len(list((multi_train_dir/cls).glob('*.txt'))),
          'test:', len(list((multi_test_dir/cls).glob('*.txt'))))


Multi-class folder structure ready.
Discovery files: 15
Kids train: 40 test: 20
Teen train: 40 test: 20
Adult train: 40 test: 20


In [10]:
multi_discovered = discover_features_from_texts(
    texts_or_file=multi_disc_dir,
    provider=provider,
    as_set=True,
    output_dir=multi_output_dir,
    output_filename='discovered_text_features_multiclass.json',
)
print('Discovered features for 3-class task:')
print(json.dumps(multi_discovered, indent=2))

Features saved to netflix_multiclass_data/outputs/discovered_text_features_multiclass.json
Discovered features for 3-class task:
{
  "features": ""
}


In [11]:
mc_train_paths = generate_features_from_texts(
    root_folder=multi_train_dir,
    discovered_features_path=multi_output_dir / 'discovered_text_features_multiclass.json',
    provider=provider,
    classes=multi_classes,
    output_dir=multi_output_dir / 'train_gen',
    merge_to_single_csv=True,
    merged_csv_name='train_feature_values.csv',
)
mc_test_paths = generate_features_from_texts(
    root_folder=multi_test_dir,
    discovered_features_path=multi_output_dir / 'discovered_text_features_multiclass.json',
    provider=provider,
    classes=multi_classes,
    output_dir=multi_output_dir / 'test_gen',
    merge_to_single_csv=True,
    merged_csv_name='test_feature_values.csv',
)

mc_train_gen = pd.read_csv(mc_train_paths['__merged__'])
mc_test_gen  = pd.read_csv(mc_test_paths ['__merged__'])
print('Train shape:', mc_train_gen.shape)
mc_train_gen.head()

Adult: 100%|██████████| 20/20 [00:25<00:00,  1.27s/file]

Train shape: (120, 3)


,File,Class,raw_llm_output
0,Kids_train_1.txt,Kids,"{""features"": {}}"
1,Kids_train_10.txt,Kids,"{""features"": {}}"
2,Kids_train_11.txt,Kids,"{""features"": {}}"
3,Kids_train_12.txt,Kids,"{""features"": {}}"
4,Kids_train_13.txt,Kids,"{""features"": {}}"


In [12]:
import ast

def expand_llm_output(df):
    """Unpack raw_llm_output into feature columns if no feature cols exist."""
    if 'raw_llm_output' not in df.columns:
        return df
    rows = []
    for _, row in df.iterrows():
        try:
            parsed = json.loads(row['raw_llm_output'])
        except Exception:
            try:
                parsed = ast.literal_eval(str(row['raw_llm_output']))
            except Exception:
                parsed = {}
        if isinstance(parsed, dict) and 'features' in parsed:
            feats = parsed['features']
            if isinstance(feats, dict):
                parsed = feats
            elif isinstance(feats, list):
                parsed = {f: 'yes' for f in feats}
            else:
                parsed = {}
        elif not isinstance(parsed, dict):
            parsed = {}
        rows.append(parsed)
    feat_df = pd.DataFrame(rows, index=df.index)
    class_col = next((c for c in df.columns if c.lower() == 'class'), 'Class')
    return pd.concat([df[[class_col]].reset_index(drop=True),
                      feat_df.reset_index(drop=True)], axis=1)

mc_train_exp = expand_llm_output(mc_train_gen)
mc_test_exp  = expand_llm_output(mc_test_gen)

print('Columns after expand:', mc_train_exp.columns.tolist())
display(mc_train_exp.head(3))

class_col = next((c for c in mc_train_exp.columns if c.lower() == 'class'), 'Class')
feat_cols  = [c for c in mc_train_exp.columns if c != class_col]

if not feat_cols:
    print('WARNING: No features found — using text_length as dummy feature')
    mc_train_exp['text_length'] = multi_train['text'].str.len().values
    mc_test_exp ['text_length'] = multi_test ['text'].str.len().values
    feat_cols = ['text_length']

mc_train_X = pd.get_dummies(mc_train_exp[feat_cols], dtype=int)
mc_test_X  = pd.get_dummies(mc_test_exp [feat_cols], dtype=int)
mc_train_X, mc_test_X = mc_train_X.align(mc_test_X, join='outer', axis=1, fill_value=0)

mc_y_train = mc_train_exp[class_col].astype(str)
mc_y_test  = mc_test_exp [class_col].astype(str)

mc_tree = DecisionTreeClassifier(max_depth=4, random_state=random_seed)
mc_tree.fit(mc_train_X, mc_y_train)
mc_pred = mc_tree.predict(mc_test_X)

print('3-class classification report:')
print(classification_report(mc_y_test, mc_pred, digits=3))


Columns after expand: ['Class']


,Class
0,Kids
1,Kids
2,Kids


3-class classification report:
              precision    recall  f1-score   support

       Adult      0.308     0.200     0.242        20
        Kids      0.368     0.700     0.483        20
        Teen      0.111     0.050     0.069        20

    accuracy                          0.317        60
   macro avg      0.262     0.317     0.265        60
weighted avg      0.262     0.317     0.265        60



In [13]:
# tests/test_multiclass.py

MULTICLASS_TESTS = '''
"""Tests for generalised N-class support."""
import json, tempfile
from pathlib import Path
from unittest.mock import MagicMock, patch
import pandas as pd
import pytest

THREE = ["Kids", "Teen", "Adult"]
FEATS = ["has_violence", "suitable_for_children", "has_romance"]

def test_generate_processes_all_class_folders(tmp_path):
    from llm_feature_gen.generate import generate_features_from_texts
    feats_path = tmp_path / "features.json"
    feats_path.write_text(json.dumps(FEATS))
    root = tmp_path / "texts"
    for cls in THREE:
        d = root / cls
        d.mkdir(parents=True)
        (d / "sample.txt").write_text(f"Sample for {cls}.", encoding="utf-8")
    dummy = {"File": "sample.txt", "Class": "Kids", **{f: "yes" for f in FEATS}}
    with patch("llm_feature_gen.generate._score_single_text", return_value=dummy):
        result = generate_features_from_texts(
            root_folder=root, discovered_features_path=feats_path,
            provider=MagicMock(), classes=THREE,
            output_dir=tmp_path / "out", merge_to_single_csv=True,
        )
        assert len(pd.read_csv(result["__merged__"])) == 3

def test_four_class_discovery(tmp_path):
    from llm_feature_gen.discover import discover_features_from_texts
    d = tmp_path / "discover"
    d.mkdir()
    for i, cls in enumerate(["A", "B", "C", "D"]):
        (d / f"{cls}_{i}.txt").write_text(f"Sample for {cls}.", encoding="utf-8")
    with patch("llm_feature_gen.discover._call_llm", return_value=json.dumps(FEATS)):
        result = discover_features_from_texts(
            texts_or_file=d, provider=MagicMock(),
            classes=["A", "B", "C", "D"], output_dir=tmp_path / "out",
        )
        assert isinstance(result, list) and len(result) > 0
'''

print(MULTICLASS_TESTS)


"""Tests for generalised N-class support."""
import json, tempfile
from pathlib import Path
from unittest.mock import MagicMock, patch
import pandas as pd
import pytest

THREE = ["Kids", "Teen", "Adult"]
FEATS = ["has_violence", "suitable_for_children", "has_romance"]

def test_generate_processes_all_class_folders(tmp_path):
    from llm_feature_gen.generate import generate_features_from_texts
    feats_path = tmp_path / "features.json"
    feats_path.write_text(json.dumps(FEATS))
    root = tmp_path / "texts"
    for cls in THREE:
        d = root / cls
        d.mkdir(parents=True)
        (d / "sample.txt").write_text(f"Sample for {cls}.", encoding="utf-8")
    dummy = {"File": "sample.txt", "Class": "Kids", **{f: "yes" for f in FEATS}}
    with patch("llm_feature_gen.generate._score_single_text", return_value=dummy):
        result = generate_features_from_texts(
            root_folder=root, discovered_features_path=feats_path,
            provider=MagicMock(), classes=THREE,
   

---
## Submission

| Item | Location |
|---|---|
| Fork | https://github.com/tch-Teutonic/LLM-based-feature-generation |
| PR — sklearn Transformer | *(open after adding `sklearn_transformer.py` + tests to fork)* |
| PR — Batch processing | *(open after adding `batch_generate.py` + tests to fork)* |
| PR — Multi-class | *(open after generalising discover/generate + tests in fork)* |